# HelioAI in three minutes — one shock, end to end

One question at a time, HelioAI finds the data, downloads it, runs the analysis in a
sandbox, and hands back a figure, a number, and a notebook you can re-run. This tour does
that once, on a single well-documented event, and **checks the number against two
independent shock databases** at the end.

**The event.** The interplanetary shock of **17 March 2015, ~04:00 UT**, seen by Wind at
L1 — the driver of the St Patrick's Day geomagnetic storm (Dst ≈ −223 nT). A strong,
isolated, fast-forward shock: the textbook case for the coplanarity method.

**What you need.** HelioAI installed, one LLM provider key in `.env`, and the parameter
index built (`helioai index`). Data access itself needs no credentials.

Each `%%helioai` cell is one turn of the agent. Expect 20–60 s per turn; the activity
lines show what it is doing meanwhile.

In [ ]:
%load_ext helioai.interfaces.jupyter_magic

import os

print("Provider:", os.environ.get("HELIOAI_LLM_PROVIDER", "opencode"))

---
## 1 — Find the parameter

You describe the quantity; HelioAI resolves the speasy id over 83 000 products (hybrid
semantic + lexical search). Watch for the **parameter card**: mission, instrument, units,
cadence, coverage.

In [ ]:
%%helioai
Which speasy parameter gives the Wind magnetic field vector in GSM coordinates
at the highest cadence available (3 s) from the MFI instrument?

---
## 2 — Download and look at the shock

The download blanks the dataset's declared fill value to NaN *before* anything else sees
the array, scans for gaps and outliers, and saves the full-resolution series in the
session. The plot comes from a sandboxed Python run; the generated script is kept.

This step is deliberately a plot and nothing else — the physics comes next, with the
recipe. On the first live run the agent reached for a second recipe here to pick its
averaging windows; the prompt now says not to.

In [ ]:
%%helioai
Download that parameter from 2015-03-17T03:30 to 2015-03-17T04:30 and plot |B| and Bz
in two panels. Mark the time of the sharp jump in |B| near 04:00 UT — that is the shock.
Plot only: do not compute upstream/downstream averages or load any recipe yet.

---
## 3 — Compute θ_Bn with the vetted recipe

θ_Bn is the angle between the upstream field and the shock normal. HelioAI ships a
recipe for it (`theta_bn`, magnetic coplanarity — Colburn & Sonett 1966; Schwartz 1998)
with a citation and a self-test. The agent is asked to **load and call it**, not to
rewrite the formula from memory — the recipe check on the answer says whether it did.

The averaging windows below follow the Harvard-CfA convention the recipe adopts by
default: 13 minutes on each side — 260 samples of 3-second data — two minutes clear of
the ramp.

In [ ]:
%%helioai
Compute theta_Bn for this shock with the theta_bn recipe. Average the upstream field
over 03:45–03:58 UT and the downstream field over 04:02–04:15 UT, call the recipe's
theta_bn() function on those two mean vectors, and export the angle (in deg), the
normal, the two mean |B| (in nT) and the magnetic compression ratio |B_dn|/|B_up|.
State which method you used and quote the angle with its unit.

---
## 4 — Check against two independent references

HelioAI certifies where a number **came from** (the provenance line under the answer
traces it to the script that computed it). Whether the number is **right** is a separate
question, and it is answered here by databases that computed the same shock without us:

| Reference | Method | θ_Bn | Compression |B_dn|/|B_up| | V_shock |
|---|---|---|---|---|
| Harvard-CfA Wind shock database, event `wi_00677` | Magnetic coplanarity (MC) — the recipe's method | **58.8 ± 2.7°** | 2.45 ± 0.46 (RH08) | 476.7 km/s (MC) |
| Harvard-CfA, same event | Rankine–Hugoniot RH08 (their preferred) | 66.1 ± 3.3° | — | 572.7 km/s |
| IPShocks (Univ. Helsinki), Wind 2015-03-17 04:00:03 | Mixed-mode (MX3) | 63.1 ± 16.8° | 2.52 ± 0.13 | 561.7 ± 39.1 km/s |

**What to expect.** The recipe applies magnetic coplanarity, so the CfA MC value is the
like-for-like comparison: an angle within **≈ 55–66°** is consistent with both databases
given their spread; the compression ratio should sit around **2.4–2.6**. A few degrees
of difference come from the exact averaging windows — the agent's answer names them, so
you can move them and see the sensitivity yourself.

Sources: CfA — <https://lweb.cfa.harvard.edu/shocks/wi_data/00677/wi_00677.html> ·
IPShocks — Kilpua et al. (2015), *JGR Space Physics* 120, doi:10.1002/2015JA021138,
database as open CSV: doi:10.5281/zenodo.19730292 · Method — Schwartz (1998), ISSI
SR-001 ch. 10, <https://www.issibern.ch/wp-content/uploads/SR-001.pdf>.

The values above were read on 2026-09-11; both databases are maintained and may be
updated.

---
## 5 — Take the analysis away

Every sandbox run of this session is saved. The export rewrites them into a standalone
notebook: `load_data()` becomes a `fetch_series()` call that redoes the fill-value
blanking exactly as the session did, the helpers are inlined, and a *Methods & data
acknowledgements* cell lists the recipe and its reference. It runs in a plain Jupyter
kernel with no HelioAI installed.

In [ ]:
%helioai_export

---
## 6 — The whole thing in one question

Steps 1–3 were one request each so you could watch every tool call. Asked as a single
question, the lead agent **plans first** and **delegates**: parameter discovery, data
analysis and literature search each go to a specialised sub-agent with its own tool
whitelist and turn budget. The activity lines show the plan, the sub-agents spawning,
and what each one returned.

This runs in a **new session** so it starts from nothing — the previous one, with the
analysis you just exported, is kept.

In [ ]:
%helioai_session new

In [ ]:
%%helioai
For the interplanetary shock seen by Wind around 04:00 UT on 2015-03-17:
find the 3-second MFI magnetic field in GSM, download 03:30–04:30 UT, plot |B| and Bz
with the shock marked, then compute theta_Bn with the theta_bn recipe using 13-minute
upstream and downstream averages clear of the ramp, and report the angle, the normal
and the compression ratio. Finally, find two peer-reviewed papers on this shock or the
storm it drove, with bibcodes.

Compare the θ_Bn with step 3 and with the two databases above. The windows here were
left to the agent ("13-minute, clear of the ramp") rather than fixed to the second, so a
degree or two of difference is the sensitivity to that choice — not an error.